# CSCI 4150 · Meeting 14 Optional Extension Notebook
## RAG: Build an Evidence Pipeline Beyond RAG Play

**Central question:** What changes when an answer must show its evidence?

This notebook builds a deliberately small, inspectable RAG pipeline using **TF-IDF + cosine similarity**. There is no API key and no hidden vector database. The goal is to see how chunking, query wording, top-k, and distractor documents change what evidence becomes visible to an answer.

Use **RAG Play first** for the main class activity. This notebook is the optional extension when you want custom documents, controlled distractors, explicit similarity scores, and citation IDs.


## What to save as evidence

For each experiment, record:

| Change made | Retrieved evidence | Answer / citation | Failure label | Interpretation |
|---|---|---|---|---|
| chunking / query / top-k / distractor | chunk IDs + scores | supported or unsupported? | R1 / R2 / R3 / G1 / C1 / E1 | what the evidence means |

A justified **reject** or **defer** decision is valid evidence.


In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Tiny source corpus. Each document has an ID so provenance survives retrieval.
documents = [
    {"source_id":"S1", "title":"Late Work Policy", "text":"Late work is accepted up to 48 hours after the deadline with a 10 percent deduction. Work more than 48 hours late requires prior instructor approval."},
    {"source_id":"S2", "title":"Regrade Policy", "text":"Regrade requests must be submitted within seven days of the grade release and must identify the rubric criterion being questioned."},
    {"source_id":"S3", "title":"Collaboration Policy", "text":"Students may discuss concepts and debugging strategies, but submitted code and written explanations must be their own unless an assignment explicitly allows shared work."},
    {"source_id":"S4", "title":"Project Milestones", "text":"Project proposal revisions are due one week after feedback. Teams should preserve evidence of design decisions and evaluation results in the project dossier."},
]

# Optional distractor: semantically similar words, different rule.
distractor = {"source_id":"D1", "title":"Old Draft Policy", "text":"An archived draft stated that late work could be submitted for seven days without penalty. This draft is obsolete and must not be used as current policy."}

pd.DataFrame(documents)


,source_id,title,text
0,S1,Late Work Policy,Late work is accepted up to 48 hours after the deadline with a 10 percent deduction. Work more than 48 hours late re...
1,S2,Regrade Policy,Regrade requests must be submitted within seven days of the grade release and must identify the rubric criterion bei...
2,S3,Collaboration Policy,"Students may discuss concepts and debugging strategies, but submitted code and written explanations must be their ow..."
3,S4,Project Milestones,Project proposal revisions are due one week after feedback. Teams should preserve evidence of design decisions and e...


In [ ]:
def chunk_documents(docs, chunk_words=26, overlap_words=6):
    chunks=[]
    step=max(1, chunk_words-overlap_words)
    for doc in docs:
        words=doc["text"].split()
        for start in range(0, len(words), step):
            piece=words[start:start+chunk_words]
            if not piece: break
            chunks.append({
                "chunk_id": f'{doc["source_id"]}_C{len([c for c in chunks if c["source_id"]==doc["source_id"]])+1}',
                "source_id": doc["source_id"],
                "title": doc["title"],
                "text": " ".join(piece)
            })
            if start + chunk_words >= len(words): break
    return pd.DataFrame(chunks)

def build_index(chunks_df):
    vectorizer=TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    matrix=vectorizer.fit_transform(chunks_df["text"])
    return vectorizer, matrix

def retrieve(query, chunks_df, vectorizer, matrix, top_k=3):
    q=vectorizer.transform([query])
    scores=cosine_similarity(q, matrix).ravel()
    idx=scores.argsort()[::-1][:top_k]
    out=chunks_df.iloc[idx].copy()
    out["score"]=scores[idx]
    return out[["chunk_id","source_id","title","score","text"]].reset_index(drop=True)

def extractive_answer(query, retrieved):
    # Simple evidence-preserving answer: choose the highest-scoring sentence from retrieved context.
    candidates=[]
    qterms=set(re.findall(r"[a-zA-Z]+", query.lower()))
    for _,row in retrieved.iterrows():
        for sent in re.split(r"(?<=[.!?])\s+", row["text"]):
            sterms=set(re.findall(r"[a-zA-Z]+", sent.lower()))
            overlap=len(qterms & sterms)
            candidates.append((overlap, row["score"], sent, row["chunk_id"]))
    candidates.sort(reverse=True)
    _,_,sent,cid=candidates[0]
    return f"{sent} [{cid}]"

def run_rag(query, docs, chunk_words=26, overlap_words=6, top_k=3):
    chunks=chunk_documents(docs, chunk_words, overlap_words)
    vectorizer, matrix=build_index(chunks)
    retrieved=retrieve(query, chunks, vectorizer, matrix, top_k)
    answer=extractive_answer(query, retrieved)
    return chunks, retrieved, answer


## Baseline

Ask a question whose answer exists in the corpus. Inspect **which chunks** become visible to the answer. Do not evaluate only the final sentence.


In [ ]:
query = "How late can I submit an assignment, and what happens?"
chunks, retrieved, answer = run_rag(query, documents, chunk_words=26, overlap_words=6, top_k=3)
print("ANSWER:", answer)
display(retrieved)


ANSWER: Students may discuss concepts and debugging strategies, but submitted code and written explanations must be their own unless an assignment explicitly allows shared work. [S3_C1]


,chunk_id,source_id,title,score,text
0,S1_C1,S1,Late Work Policy,0.219565,Late work is accepted up to 48 hours after the deadline with a 10 percent deduction. Work more than 48 hours late re...
1,S3_C1,S3,Collaboration Policy,0.133054,"Students may discuss concepts and debugging strategies, but submitted code and written explanations must be their ow..."
2,S4_C1,S4,Project Milestones,0.000000,Project proposal revisions are due one week after feedback. Teams should preserve evidence of design decisions and e...


## Experiment 1 — Change chunking

Change **one variable**. Try `chunk_words=10` with `overlap_words=0`, then compare with the baseline.

Questions:
- Did the best evidence split across chunks?
- Did a needed condition disappear from the top-ranked chunk?
- Is this a retrieval miss (R1) or a chunk-boundary failure (R2)?


In [ ]:
# TODO: change chunk_words / overlap_words and rerun
chunks_small, retrieved_small, answer_small = run_rag(
    query, documents, chunk_words=10, overlap_words=0, top_k=3
)
print("ANSWER:", answer_small)
display(retrieved_small)


ANSWER: and written explanations must be their own unless an assignment [S3_C2]


,chunk_id,source_id,title,score,text
0,S3_C2,S3,Collaboration Policy,0.287310,and written explanations must be their own unless an assignment
1,S1_C1,S1,Late Work Policy,0.199410,Late work is accepted up to 48 hours after the
2,S1_C3,S1,Late Work Policy,0.171712,hours late requires prior instructor approval.


## Experiment 2 — Change top-k or query wording

Try `top_k=1` and `top_k=4`. Then rewrite the query using different language (for example “deadline exception” instead of “late assignment”).

Ask: does more context help, or does it add distraction?


In [ ]:
query2 = "What is the deadline exception for submitted work?"
chunks2, retrieved2, answer2 = run_rag(query2, documents, chunk_words=26, overlap_words=6, top_k=1)
print("ANSWER:", answer2)
display(retrieved2)


ANSWER: Late work is accepted up to 48 hours after the deadline with a 10 percent deduction. [S1_C1]


,chunk_id,source_id,title,score,text
0,S1_C1,S1,Late Work Policy,0.23253,Late work is accepted up to 48 hours after the deadline with a 10 percent deduction. Work more than 48 hours late re...


## Experiment 3 — Add a distractor

Now include an obsolete policy containing highly relevant words. A retrieval system can rank it highly even though it is not authoritative.

If the distractor enters context, label the failure **R3**. Then ask what metadata or filtering rule would prevent it.


In [ ]:
docs_with_distractor = documents + [distractor]
chunks3, retrieved3, answer3 = run_rag(query, docs_with_distractor, chunk_words=26, overlap_words=6, top_k=3)
print("ANSWER:", answer3)
display(retrieved3)


ANSWER: Students may discuss concepts and debugging strategies, but submitted code and written explanations must be their own unless an assignment explicitly allows shared work. [S3_C1]


,chunk_id,source_id,title,score,text
0,S1_C1,S1,Late Work Policy,0.165927,Late work is accepted up to 48 hours after the deadline with a 10 percent deduction. Work more than 48 hours late re...
1,S3_C1,S3,Collaboration Policy,0.146241,"Students may discuss concepts and debugging strategies, but submitted code and written explanations must be their ow..."
2,D1_C1,D1,Old Draft Policy,0.108395,An archived draft stated that late work could be submitted for seven days without penalty. This draft is obsolete an...


## Evidence check — relevance, support, citation alignment

For the answer you just produced, check three things separately:
1. **Relevance:** are the retrieved chunks about the question?
2. **Support:** does the cited text justify the answer claim?
3. **Citation alignment:** does the bracketed chunk ID actually point to the supporting evidence?


In [ ]:
evidence = pd.DataFrame([
    {"change_made":"baseline", "retrieved_evidence":", ".join(retrieved["chunk_id"]), "answer_citation":answer, "failure_label":"TBD", "interpretation":"TBD"},
    {"change_made":"smaller chunks / no overlap", "retrieved_evidence":", ".join(retrieved_small["chunk_id"]), "answer_citation":answer_small, "failure_label":"TBD", "interpretation":"TBD"},
    {"change_made":"distractor added", "retrieved_evidence":", ".join(retrieved3["chunk_id"]), "answer_citation":answer3, "failure_label":"TBD", "interpretation":"TBD"},
])
evidence


,change_made,retrieved_evidence,answer_citation,failure_label,interpretation
0,baseline,"S1_C1, S3_C1, S4_C1","Students may discuss concepts and debugging strategies, but submitted code and written explanations must be their ow...",TBD,TBD
1,smaller chunks / no overlap,"S3_C2, S1_C1, S1_C3",and written explanations must be their own unless an assignment [S3_C2],TBD,TBD
2,distractor added,"S1_C1, S3_C1, D1_C1","Students may discuss concepts and debugging strategies, but submitted code and written explanations must be their ow...",TBD,TBD


## Project transfer

Before you leave, answer:
1. What external/changing/private knowledge would your project need?
2. Which sources are trustworthy enough to retrieve from?
3. Which retrieval variable would you test first?
4. What would count as success?
5. What would count as failure?
6. Based on today, should your team **adopt**, **modify**, **reject**, or **defer** RAG? Why?


In [ ]:
project_plan = {
    "knowledge_need":"",
    "sources":"",
    "retrieval_variable":"chunking / representation / top-k / metadata / query wording",
    "success_criterion":"",
    "failure_criterion":"",
    "decision":"adopt / modify / reject / defer",
    "why":""
}
project_plan


{'knowledge_need': '',
 'sources': '',
 'retrieval_variable': 'chunking / representation / top-k / metadata / query wording',
 'success_criterion': '',
 'failure_criterion': '',
 'decision': 'adopt / modify / reject / defer',
 'why': ''}